# Desafio 4 — Ciência de Dados 1

**Aluno(a): Gabriel Melo dos Santos**

Conteúdo das **Aulas 08 a 11**: busca em grade, busca aleatória, curva de validação e curva
de aprendizado.

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa escrever
código — o objetivo é interpretar e comparar os resultados. Os arquivos `.csv` estão nesta
pasta.

> **Regra deste desafio:** toda afirmação que menciona um valor deve **trazer o valor**,
> copiado da saída da sua célula. Os números mudam de caderno para caderno.

Algumas células rodam várias buscas e curvas, e podem levar um ou dois minutos. Espere o
`[*]` virar número antes de passar adiante.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import loguniform, randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, validation_curve,
                                     learning_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def logistica(**kw):
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000, **kw))])

def arvore(**kw):
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", DecisionTreeClassifier(random_state=0, **kw))])

cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("ambiente pronto")

## Exercício 1 — As duas assinaturas da curva de aprendizado  *(2,0 pontos)*

Na base `q_aprendizado.csv`, dois modelos passam pela mesma curva de aprendizado: o **A** é
uma árvore sem limite de profundidade e o **B** é uma árvore de um nó só.

**(a)** Qual dos dois tem **viés alto** e qual tem **variância alta**? Justifique com o treino,
a validação e o gap de cada um no maior tamanho de treino.

**(b)** Explique por que o gap é o sinal que separa os dois casos, e não a nota de validação
sozinha. Use os valores dos dois modelos.

**(c)** Em qual dos dois **mais dados** tem chance de ajudar? Diga por quê.

**(d)** E no outro, o que você faria? Dê duas ações concretas e diga o que cada uma muda.

In [ ]:
df = pd.read_csv("q_aprendizado.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1])

frac = np.linspace(0.2, 1.0, 5)
for nome, modelo in (("A — árvore profunda", arvore(max_depth=None)),
                     ("B — árvore de 1 nó ", arvore(max_depth=1))):
    tam, tr, va = learning_curve(modelo, X, y, train_sizes=frac, cv=cv, scoring="roc_auc")
    a, b = tr.mean(axis=1), va.mean(axis=1)
    print(f"\n{nome}")
    print("  exemplos  treino  validação     gap")
    for t, u, v in zip(tam, a, b):
        print(f"  {t:8d}  {u:.4f}   {v:.4f}   {u - v:.4f}")
    if nome.startswith("A"):
        A_tr, A_va = a, b
    else:
        B_tr, B_va = a, b

print(f"\nno maior tamanho — A: treino {A_tr[-1]:.4f} validação {A_va[-1]:.4f} "
      f"gap {A_tr[-1] - A_va[-1]:.4f}")
print(f"                   B: treino {B_tr[-1]:.4f} validação {B_va[-1]:.4f} "
      f"gap {B_tr[-1] - B_va[-1]:.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 2 — As três regiões da curva de validação  *(1,5 pontos)*

A base `q_curva.csv` alimenta uma curva de validação de uma árvore, variando a profundidade
de 1 a 12. A saída traz treino, validação e o **gap** (treino − validação) em cada ponto.

**(a)** Aponte as três regiões da curva — subajuste, ótimo e sobreajuste — dizendo em que
profundidades cada uma está e citando os valores de validação que sustentam a sua divisão.

**(b)** Compare o gap na profundidade 1 com o gap na profundidade 12. Dê os dois valores e
explique o que o crescimento do gap significa.

**(c)** Qual profundidade você entrega, e por quê? Cite a nota de validação dela.

**(d)** Por que a decisão se lê na linha de **validação** e não na de treino? Use os números
da profundidade 12 na sua resposta.

In [ ]:
df = pd.read_csv("q_curva.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1])

prof = list(range(1, 13))
tr, va = validation_curve(arvore(), X, y, param_name="clf__max_depth",
                          param_range=prof, cv=cv, scoring="roc_auc")
m_tr, m_va = tr.mean(axis=1), va.mean(axis=1)

print("\nprof  treino  validação     gap")
for p, a, b in zip(prof, m_tr, m_va):
    print(f"{p:4d}  {a:.4f}   {b:.4f}   {a - b:.4f}")

i = int(np.argmax(m_va))
print(f"\npico da validação: profundidade {prof[i]}  nota {m_va[i]:.4f}")
print(f"gap na profundidade  1: {m_tr[0] - m_va[0]:.4f}")
print(f"gap na profundidade 12: {m_tr[-1] - m_va[-1]:.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 3 — A grade e a borda da faixa  *(2,0 pontos)*

A base `q_grade.csv` passa por duas buscas em grade sobre o mesmo modelo: uma varre quatro
valores de `C` e a outra varre sete, acrescentando valores **menores** (regularização mais
forte).

**(a)** Qual `C` vence em cada grade, e com que nota? Dê os quatro valores.

**(b)** Olhe onde caiu o vencedor da grade curta. O que a posição dele dentro da faixa indica,
e por que isso é um aviso e não um resultado?

**(c)** Quanto se ganhou ao ampliar a faixa? Dê o valor e diga se, no seu caso, valeu a pena.

**(d)** Quantos ajustes de modelo cada busca custou, e como esse custo cresce se você
acrescentar um segundo hiperparâmetro com cinco valores à grade larga?

In [ ]:
df = pd.read_csv("q_grade.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1], "| positivos:", f"{y.mean():.1%}")

curta = [0.1, 1, 10, 100]
larga = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]

bc = GridSearchCV(logistica(), {"clf__C": curta}, cv=cv, scoring="roc_auc").fit(X, y)
bl = GridSearchCV(logistica(), {"clf__C": larga}, cv=cv, scoring="roc_auc").fit(X, y)

print("\nGRADE CURTA")
for C, m in zip(curta, bc.cv_results_["mean_test_score"]):
    print(f"  C = {C:<8} -> {m:.4f}")
print(f"  melhor: C = {bc.best_params_['clf__C']}  nota {bc.best_score_:.4f}")

print("\nGRADE LARGA")
for C, m in zip(larga, bl.cv_results_["mean_test_score"]):
    print(f"  C = {C:<8} -> {m:.4f}")
print(f"  melhor: C = {bl.best_params_['clf__C']}  nota {bl.best_score_:.4f}")

print(f"\najustes: curta {len(curta) * 5}  |  larga {len(larga) * 5}")
print(f"ganho ao ampliar: {bl.best_score_ - bc.best_score_:+.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 4 — Pico ou platô?  *(1,5 pontos)*

A mesma ideia da questão anterior, agora sobre o `C` de uma regressão logística na base
`q_plato.csv` — mas olhando também o **desvio entre as dobras** de cada ponto.

**(a)** Qual `C` tem a maior nota de validação, e qual o desvio nesse ponto? Dê os dois
valores.

**(b)** Quais valores de `C` ficam dentro da faixa de empate (a nota do pico menos um desvio
dele)? Liste-os com as notas.

**(c)** Qual valor você entrega, e por quê? A resposta não é necessariamente o pico — diga qual
é o critério de desempate.

**(d)** Se você reportasse apenas a nota do pico, sem o desvio, que informação o leitor
perderia?

In [ ]:
df = pd.read_csv("q_plato.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1])

valores = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
tr, va = validation_curve(logistica(), X, y, param_name="clf__C",
                          param_range=valores, cv=cv, scoring="roc_auc")
m, s = va.mean(axis=1), va.std(axis=1)

print("\n     C   validação    desvio")
for c, a, b in zip(valores, m, s):
    print(f"{c:>6}    {a:.4f}    {b:.4f}")

i = int(np.argmax(m))
piso = m[i] - s[i]
dentro = [c for c, a in zip(valores, m) if a >= piso]
print(f"\npico: C = {valores[i]}  nota {m[i]:.4f} ± {s[i]:.4f}")
print(f"piso da faixa de empate: {piso:.4f}")
print(f"valores dentro da faixa: {dentro}")

**Resposta:**

_(escreva aqui)_

## Exercício 5 — Grade e sorteio com o mesmo orçamento  *(1,5 pontos)*

Na base `q_sorteio.csv`, uma busca em grade varre todas as combinações de dois
hiperparâmetros de uma árvore. Em seguida, uma busca aleatória sorteia **o mesmo número** de
combinações — repetida com cinco sementes diferentes.

**(a)** Qual a nota da grade e quais as cinco notas do sorteio? Dê a média, o mínimo e o
máximo do sorteio.

**(b)** A grade ganhou do sorteio? Compare a diferença entre as duas com a **amplitude** entre
as sementes, e diga o que essa comparação permite concluir.

**(c)** O que faz o custo de cada uma das duas buscas crescer quando você acrescenta mais um
hiperparâmetro ao espaço?

**(d)** Em que situação você escolheria o sorteio em vez da grade? Justifique com o que você
observou aqui.

In [ ]:
df = pd.read_csv("q_sorteio.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1])

grade = {"clf__max_depth": [2, 4, 6, 10],
         "clf__min_samples_leaf": [1, 5, 10, 20]}          # 16 combinações
g = GridSearchCV(arvore(), grade, cv=cv, scoring="roc_auc").fit(X, y)
print(f"\nGRADE  ({len(g.cv_results_['params'])} combinações)")
print("  melhor:", g.best_params_, f"nota {g.best_score_:.4f}")

print("\nSORTEIO (16 amostras, cinco sementes)")
espaco = {"clf__max_depth": randint(2, 13),
          "clf__min_samples_leaf": randint(1, 21)}
notas = []
for s in range(5):
    r = RandomizedSearchCV(arvore(), espaco, n_iter=16, cv=cv, scoring="roc_auc",
                           random_state=s).fit(X, y)
    notas.append(r.best_score_)
    print(f"  semente {s}: {r.best_score_:.4f}   {r.best_params_}")

notas = np.array(notas)
print(f"\nsorteio: média {notas.mean():.4f}  mínimo {notas.min():.4f}  "
      f"máximo {notas.max():.4f}  amplitude {notas.max() - notas.min():.4f}")
print(f"grade - média do sorteio: {g.best_score_ - notas.mean():+.4f}")

**Resposta:**

_(escreva aqui)_

## Exercício 6 — Vale a pena coletar mais dados?  *(1,5 pontos)*

A base `q_coleta.csv` passa por uma curva de aprendizado com seis tamanhos de treino. O
cliente quer saber se vale pagar por mais uma rodada de coleta.

**(a)** Dê o ganho do primeiro passo e o do último. O que a comparação entre os dois mostra?

**(b)** Compare o ganho do último passo com o **desvio** daquele ponto. Dê os dois valores e
diga se o ganho está demonstrado.

**(c)** Responda ao cliente em uma frase, citando pelo menos um número.

**(d)** Se mais dados não é o caminho, onde você sugeriria investir? Dê duas alternativas
ligadas ao que a curva mostra.

In [ ]:
df = pd.read_csv("q_coleta.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("linhas:", len(y), "| colunas:", X.shape[1])

frac = np.linspace(0.1, 1.0, 6)
tam, tr, va = learning_curve(logistica(), X, y, train_sizes=frac, cv=cv, scoring="roc_auc")
m, s = va.mean(axis=1), va.std(axis=1)

print("\nexemplos  validação   desvio     ganho")
for k, (t, a, b) in enumerate(zip(tam, m, s)):
    g = "   —" if k == 0 else f"{a - m[k-1]:+.4f}"
    print(f"{t:8d}   {a:.4f}   {b:.4f}   {g}")

print(f"\nganho do primeiro passo: {m[1] - m[0]:+.4f}")
print(f"ganho do último passo  : {m[-1] - m[-2]:+.4f}")
print(f"desvio do último ponto : {s[-1]:.4f}")

**Resposta:**

_(escreva aqui)_